# 01f — MY / MY2 Baseline Investigation

**Goal**: Understand how the baseline is determined for the MY and MY2 datasets,
and investigate whether the baseline method affects the decomposition results.

**Strategy**: Use **untrimmed** data for early analysis to avoid accidentally
discarding relevant signal (e.g. the minor peak near frame 1200 seen in 01d).

**Reference**: [Tutorial Chapter 4 — Baseline Correction](https://biosaxs-dev.github.io/molass-tutorial/chapters/04/data_correction.html)

**Key API**:
- `ssd.plot_compact(baseline=True)` — visualise where the baseline is drawn (works on untrimmed data too)
- `ssd.set_baseline_method('linear')` — default
- `ssd.set_baseline_method(('linear', 'uvdiff'))` — linear XR + UV-difference UV
- `ssd.set_baseline_method('integral')` — integral method
- `ssd.get_baseline_method()` — confirm active method
- `ssd.corrected_copy()` — apply correction

---
**Discussion plan (step by step)**:
1. Visualise default (linear) baseline for MY and MY2 on **untrimmed** data
2. Compare `uvdiff` UV baseline for both
3. Compare `integral` baseline for both
4. Decide on trimming + baseline strategy based on what we see


In [ ]:
%matplotlib inline
from molass import get_version
assert get_version() >= "0.8.5", f"Need >= 0.8.5, got {get_version()}"
print("molass", get_version())

import os
DATA_ROOT = r"C:\Users\takahashi\Dropbox\MOLASS\DATA\20260305"
print("Data root:", DATA_ROOT)
print("Exists:", os.path.exists(DATA_ROOT))

UV_SIGNAL   = 290   # nm — ATP/MY/MY2 signal wavelength
UV_BASELINE = 400   # nm — baseline channel


In [ ]:
from molass.DataObjects import SecSaxsData as SSD

# Use untrimmed data for early baseline investigation
ssd_my  = SSD(os.path.join(DATA_ROOT, "MY"),  uv_pickat=UV_SIGNAL)
ssd_my2 = SSD(os.path.join(DATA_ROOT, "MY2"), uv_pickat=UV_SIGNAL)

print("MY  XR shape:", ssd_my.xr.M.shape)
print("MY2 XR shape:", ssd_my2.xr.M.shape)


## Step 1 — Default (linear) baseline (untrimmed)

Visualise the default linear baseline on the full untrimmed data.
This shows the entire XR and UV elution profile including any shoulder or minor peak regions.


In [ ]:
# MY — default linear baseline, untrimmed
ssd_my.plot_compact(baseline=True)


In [ ]:
# MY2 — default linear baseline, untrimmed
ssd_my2.plot_compact(baseline=True)


## Observations — Step 1

*(Fill in after viewing the plots above.)*

- **MY**:
- **MY2**:

---
## Step 1b — Investigating the negative UV dip

The `plot_compact` UV curve is the raw instrument signal at 290 nm — no baseline subtraction
has been applied. Let us plot both the 290 nm (signal) and 400 nm (baseline reference)
channels to understand what the detector actually recorded.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col, (ssd, name) in enumerate([(ssd_my, "MY"), (ssd_my2, "MY2")]):
    # Extract UV elution curves at both wavelengths
    uv_sig = ssd.uv.get_icurve(pickat=UV_SIGNAL)    # 290 nm
    uv_bl  = ssd.uv.get_icurve(pickat=UV_BASELINE)   # 400 nm

    # Top: full range
    ax = axes[0, col]
    ax.plot(uv_sig.x, uv_sig.y, label=f"UV @ {UV_SIGNAL} nm (signal)", color="C0")
    ax.plot(uv_bl.x,  uv_bl.y,  label=f"UV @ {UV_BASELINE} nm (baseline)", color="C3", alpha=0.7)
    ax.axhline(0, color="gray", ls="--", lw=0.5)
    ax.axvline(1200, color="green", ls=":", alpha=0.5, label="frame 1200")
    ax.set_title(f"{name} — UV channels (full range)")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Absorbance (raw)")
    ax.legend(fontsize=8)

    # Bottom: zoom around frame 1000–1400
    ax = axes[1, col]
    mask = (uv_sig.x >= 1000) & (uv_sig.x <= 1400)
    ax.plot(uv_sig.x[mask], uv_sig.y[mask], label=f"UV @ {UV_SIGNAL} nm", color="C0")
    ax.plot(uv_bl.x[mask],  uv_bl.y[mask],  label=f"UV @ {UV_BASELINE} nm", color="C3", alpha=0.7)
    ax.axhline(0, color="gray", ls="--", lw=0.5)
    ax.axvline(1200, color="green", ls=":", alpha=0.5)
    ax.set_title(f"{name} — zoom frames 1000–1400")
    ax.set_xlabel("Frame")
    ax.set_ylabel("Absorbance (raw)")
    ax.legend(fontsize=8)

    # Print key stats
    neg_mask = uv_sig.y < 0
    if np.any(neg_mask):
        first_neg = uv_sig.x[neg_mask][0]
        min_val = uv_sig.y.min()
        min_frame = uv_sig.x[np.argmin(uv_sig.y)]
        print(f"{name}: first negative UV@{UV_SIGNAL}nm at frame {first_neg:.0f}, "
              f"min = {min_val:.4f} at frame {min_frame:.0f}")
    else:
        print(f"{name}: no negative values in UV@{UV_SIGNAL}nm")

fig.tight_layout()
plt.savefig("01f_uv_channels.png", dpi=150)
print("Saved: 01f_uv_channels.png")


## Observations — Step 1b (UV channels)

**What the plot shows**: Raw UV absorbance at 290 nm (signal) and 400 nm (reference baseline channel)
for both MY and MY2. Top row: full range. Bottom row: zoom into frames 1000–1400.

**Key question**: Why does the UV @ 290 nm drop below zero after the minor peak near frame 1200?

**Note**: These are raw instrument readings — no baseline subtraction by the software. Negative
absorbance means the detector output fell below its reference/blank level.

- **MY**: Shows a small positive peak followed by a negative dip, both localised around frames 1190–1340 and wavelengths 270–300 nm (near the protein absorbance band). The 400 nm channel is flat and unaffected throughout.
- **MY2**: Similar localised irregularity in the same frame/wavelength region.

---
### Observation: trimming strategy and baseline

The default trimming removes these irregular minor peaks (both the positive shoulder and the negative dip).  
**This does not worsen baseline determination** — the baseline is estimated from the flat pre- and post-peak buffer regions, which are unaffected by this localised anomaly.

The irregular peaks should be treated as a **separate UV-specific phenomenon**, distinct from:
- the main myosin elution peak (used for SAXS analysis), and
- the dc/dt-proportional baseline fluctuation documented in `baseline-fluctuation-mystery`.

Key features that distinguish this anomaly:
- **Wavelength-selective**: confined to 270–300 nm (protein absorption band); 400 nm channel is flat → not a global detector drift
- **Temporally localised**: frames ~1190–1340, after the minor peak — not a smooth bipolar ripple
- **Not derivative-shaped**: does not follow dc/dt pattern of the known baseline fluctuation mystery

**Working hypothesis**: possible inner filter effect or refractive index artifact as a concentrated aggregate slug passes through and then clears the UV detector cell.

**Action**: set aside for dedicated investigation; proceed with default trim + linear baseline for MY/MY2 decomposition analysis.


In [ ]:
# MY — 3D view (untrimmed)
ssd_my.plot_3d(title="MY — 3D (untrimmed)")


In [ ]:
# MY2 — 3D view (untrimmed)
ssd_my2.plot_3d(title="MY2 — 3D (untrimmed)")


In [ ]:
from molass.PlotUtils.MatrixPlot import simple_plot_3d

# Focus on the negative-value region in MY
WV_LO, WV_HI = 270, 300
FR_LO, FR_HI = 1190, 1340

uv = ssd_my.uv
wv = uv.iv
fr = uv.jv

wv_mask = (wv >= WV_LO) & (wv <= WV_HI)
fr_mask = (fr >= FR_LO) & (fr <= FR_HI)

M_slice  = uv.M[np.ix_(wv_mask, fr_mask)]
wv_slice = wv[wv_mask]
fr_slice = fr[fr_mask]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), subplot_kw=dict(projection='3d'))

for ax in (ax1, ax2):
    simple_plot_3d(ax, M_slice, x=wv_slice, y=fr_slice, cmap='coolwarm')
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Frame")
    ax.set_zlabel("Absorbance")

# "Frame view": look along the frame axis → see wavelength vs absorbance face-on
ax1.view_init(elev=0, azim=0)
ax1.set_title("MY — frame view (azim=0)\nwavelength profile visible")

# "Wavelength view": look along the wavelength axis → see frame vs absorbance face-on
ax2.view_init(elev=0, azim=90)
ax2.set_title("MY — wavelength view (azim=90)\nframe profile visible")

fig.suptitle(f"MY UV 3D — wavelengths {WV_LO}–{WV_HI} nm, frames {FR_LO}–{FR_HI}", fontsize=13)
fig.tight_layout()
